In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/mlp-jan-26-2025-kaggle-assignment-3/sample_submission.csv
/kaggle/input/competitions/mlp-jan-26-2025-kaggle-assignment-3/train.csv
/kaggle/input/competitions/mlp-jan-26-2025-kaggle-assignment-3/test.csv


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
!pip install vaderSentiment
 
from ydata_profiling import ProfileReport
 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer, MinMaxScaler
from scipy.sparse import hstack, csr_matrix
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
 
print("All imports done ✓")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.1 MB/s eta 0:00:00
All imports done ✓


In [3]:
train = pd.read_csv("/kaggle/input/competitions/mlp-jan-26-2025-kaggle-assignment-3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/mlp-jan-26-2025-kaggle-assignment-3/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/mlp-jan-26-2025-kaggle-assignment-3/sample_submission.csv")


print("=" * 60)
print(f"TRAIN shape      : {train.shape}")
print(f"TEST shape       : {test.shape}")
print(f"SAMPLE SUB shape : {sample_sub.shape}")
print("=" * 60)
print("\nTrain head:")
print(train.head())
print("\nTest head:")
print(test.head())
print("\nSample submission head:")
print(sample_sub.head())

TRAIN shape      : (7000, 6)
TEST shape       : (1700, 5)
SAMPLE SUB shape : (1700, 2)

Train head:
   id                                             phrase  feature_1  \
0   0  It may as well be called `` Jar-Jar Binks : Th...       14.0   
1   1                               You have to see it .        6.0   
2   2  ... either you 're willing to go with this cla...       16.0   
3   3  Watching Harris ham it up while physically and...       37.0   
4   4  Pete 's screenplay manages to find that real n...       20.0   

   feature_2  feature_3  sentiment  
0        5.0        7.0          0  
1        1.0        NaN          2  
2        0.0        6.0          1  
3        NaN        3.0          1  
4        1.0        4.0          2  

Test head:
   id                                             phrase  feature_1  \
0   0  One thing you have to give them credit for : T...        NaN   
1   1  Showtime 's starry cast could be both an asset...       13.0   
2   2  The result is good 

In [4]:
print("\n>>> Running ydata-profiling on train (this may take ~1 min) ...")
profile = ProfileReport(
    train,
    title="Sentiment Train EDA",
    explorative=True,
    minimal=False,
    progress_bar=False
)


>>> Running ydata-profiling on train (this may take ~1 min) ...


In [5]:
profile

100%|██████████| 6/6 [00:00<00:00, 22.07it/s]


<IPython.core.display.HTML object>

In [6]:
desc = profile.get_description()
 
print("\n" + "=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"  Variables             : {desc.table['n_var']}")
print(f"  Observations          : {desc.table['n']}")
print(f"  Missing cells         : {desc.table['n_cells_missing']}")
print(f"  Missing cells (%)     : {desc.table['p_cells_missing']:.2%}")
print(f"  Duplicate rows        : {desc.table['n_duplicates']}")
 
print("\n" + "=" * 60)
print("PER-VARIABLE STATS")
print("=" * 60)
for col, stats in desc.variables.items():
    print(f"\n  [ {col} ]")
    for k in ['type', 'n_missing', 'p_missing', 'n_unique']:
        if k in stats:
            val = stats[k]
            print(f"    {k:20s}: {val:.4f}" if isinstance(val, float) else f"    {k:20s}: {val}")
    for k in ['mean', 'std', 'min', 'max', 'median']:
        if k in stats:
            print(f"    {k:20s}: {stats[k]:.4f}")
 
print("\n" + "=" * 60)
print("TARGET DISTRIBUTION (sentiment)")
print("=" * 60)
vc = train['sentiment'].value_counts().sort_index()
labels = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
for lbl, count in vc.items():
    bar = '█' * int(count / vc.max() * 30)
    print(f"  {labels[lbl]:10s} ({lbl}): {count:5d}  {bar}")
 
print("\n" + "=" * 60)
print("PHRASE LENGTH STATS")
print("=" * 60)
train['word_count'] = train['phrase'].str.split().str.len()
test['word_count']  = test['phrase'].str.split().str.len()
for name, df in [('Train', train), ('Test', test)]:
    wc = df['word_count']
    print(f"  {name} → min:{wc.min()}  max:{wc.max()}  "
          f"mean:{wc.mean():.1f}  median:{wc.median():.0f}  std:{wc.std():.1f}")
 
print("\n  Mean word count by sentiment:")
print(train.groupby('sentiment')['word_count']
      .mean().rename({0:'Negative', 1:'Neutral', 2:'Positive'}).to_string())
 
print("\n" + "=" * 60)
print("NUMERIC FEATURE STATS")
print("=" * 60)
print(train[['feature_1', 'feature_2', 'feature_3']].describe().to_string())
 
print("\n" + "=" * 60)
print("MISSING VALUES — TRAIN")
print("=" * 60)
print(train.isnull().sum().to_string())
 
print("\n" + "=" * 60)
print("TRAIN / TEST PHRASE OVERLAP")
print("=" * 60)
overlap = set(train['phrase'].str.lower()) & set(test['phrase'].str.lower())
print(f"  Overlapping phrases   : {len(overlap)}")
print(f"  Duplicate train rows  : {train['phrase'].duplicated().sum()}")


DATASET OVERVIEW
  Variables             : 6
  Observations          : 7000
  Missing cells         : 3078
  Missing cells (%)     : 7.33%
  Duplicate rows        : 0

PER-VARIABLE STATS

  [ id ]
    type                : Numeric
    n_missing           : 0
    p_missing           : 0.0000
    n_unique            : 7000
    mean                : 3499.5000
    std                 : 2020.8703
    min                 : 0.0000
    max                 : 6999.0000

  [ phrase ]
    type                : Text
    n_missing           : 0
    p_missing           : 0.0000
    n_unique            : 6646

  [ feature_1 ]
    type                : Numeric
    n_missing           : 912
    p_missing           : 0.1303
    n_unique            : 2
    mean                : 19.0256
    std                 : 9.3036
    min                 : 1.0000
    max                 : 52.0000

  [ feature_2 ]
    type                : Numeric
    n_missing           : 1104
    p_missing           : 0.1577
    n_u

In [7]:
train = train.drop_duplicates(subset='phrase').reset_index(drop=True)
print(f"After dedup — train shape: {train.shape}")
 
analyzer = SentimentIntensityAnalyzer()
 
POSITIVE_WORDS = {'great','excellent','brilliant','wonderful','amazing',
                  'masterpiece','perfect','fantastic','superb','outstanding'}
NEGATIVE_WORDS = {'bad','terrible','awful','horrible','worst','boring',
                  'dreadful','disappointing','waste','pathetic'}
CONTRAST_WORDS = {'but','however','although','yet','though','while'}
 
def build_extra_features(phrases):
    out = []
    for p in phrases:
        v = analyzer.polarity_scores(p)
        w = p.lower().split()
        out.append([
            v['neg'], v['neu'], v['pos'], v['compound'],
            len(w),
            p.count('!'), p.count('?'),
            sum(1 for x in p.split() if x.isupper() and len(x) > 1),
            int(any(x in w for x in CONTRAST_WORDS)),
            int(any(x in w for x in POSITIVE_WORDS)),
            int(any(x in w for x in NEGATIVE_WORDS)),
        ])
    return np.array(out, dtype=np.float32)
 
def build_extra_features_nonneg(phrases):
    """
    Same features but scaled to [0, 1] via MinMaxScaler —
    required for ComplementNB which rejects negative values.
    Fit scaler on train phrases only; call transform for val/test.
    Returns raw array; caller must fit/transform with a MinMaxScaler.
    """
    return build_extra_features(phrases)
 
print("Building features ...")
X = train['phrase']
y = train['sentiment']
 
X_tr, X_v, y_tr, y_v = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
print(f"Train: {len(X_tr)}  |  Val: {len(X_v)}")
 
ef_tr = build_extra_features(X_tr.values)
ef_v  = build_extra_features(X_v.values)
 
# ── MinMaxScaler for non-negative extra features (CNB only) ─
mm_scaler = MinMaxScaler()
ef_tr_nonneg = mm_scaler.fit_transform(ef_tr)   # fit on train only
ef_v_nonneg  = mm_scaler.transform(ef_v)
 
# TF-IDF
vec = TfidfVectorizer(ngram_range=(1,2), max_features=80_000,
                      sublinear_tf=True, min_df=2)
W_tr = vec.fit_transform(X_tr)
W_v  = vec.transform(X_v)
 
# Full combined sparse matrix (TF-IDF + extra) — may contain negatives
X_tr_all = hstack([W_tr, csr_matrix(ef_tr)])
X_v_all  = hstack([W_v,  csr_matrix(ef_v)])
 
# Non-negative version for ComplementNB (TF-IDF ≥ 0, extra scaled to [0,1])
X_tr_nonneg = hstack([W_tr, csr_matrix(ef_tr_nonneg)])
X_v_nonneg  = hstack([W_v,  csr_matrix(ef_v_nonneg)])
 
# For KNN / Tree models that need dense: use SVD-reduced representation
svd  = TruncatedSVD(n_components=100, random_state=42)
norm = Normalizer()
X_tr_dense = norm.fit_transform(svd.fit_transform(W_tr))
X_v_dense  = norm.transform(svd.transform(W_v))
X_tr_dense_ef = np.hstack([X_tr_dense, ef_tr])
X_v_dense_ef  = np.hstack([X_v_dense,  ef_v])
 
print(f"Sparse feature dim        : {X_tr_all.shape[1]:,}")
print(f"Sparse non-neg feature dim: {X_tr_nonneg.shape[1]:,}")
print(f"Dense feature dim         : {X_tr_dense_ef.shape[1]}")
 

After dedup — train shape: (6823, 7)
Building features ...
Train: 5799  |  Val: 1024
Sparse feature dim        : 15,358
Sparse non-neg feature dim: 15,358
Dense feature dim         : 111


In [8]:
results = {}
 
#  Model 1: Logistic Regression
print("\n[1/7] Logistic Regression ...")
lr = LogisticRegression(C=1.0, max_iter=1000, solver='saga', tol=1e-3)
lr.fit(X_tr_all, y_tr)
acc = accuracy_score(y_v, lr.predict(X_v_all))
results['Logistic Regression'] = acc
print(f"      Val Accuracy: {acc:.4f}")
 
#  Model 2: SGD Classifier
print("\n[2/7] SGD Classifier ...")
sgd = SGDClassifier(loss='modified_huber', alpha=1e-4, max_iter=200,
                    random_state=42, tol=1e-3)
sgd.fit(X_tr_all, y_tr)
acc = accuracy_score(y_v, sgd.predict(X_v_all))
results['SGD Classifier'] = acc
print(f"      Val Accuracy: {acc:.4f}")
 
#  Model 3: Complement Naive Bayes
# We use X_tr_nonneg / X_v_nonneg where extra features are MinMax-scaled to [0,1].
print("\n[3/7] Complement Naive Bayes ...")
cnb = ComplementNB(alpha=0.5)
cnb.fit(X_tr_nonneg, y_tr)          # ← non-negative matrix
acc = accuracy_score(y_v, cnb.predict(X_v_nonneg))
results['Complement Naive Bayes'] = acc
print(f"      Val Accuracy: {acc:.4f}")
 
# ── Model 4: Decision Tree
print("\n[4/7] Decision Tree ...")
dt = DecisionTreeClassifier(max_depth=20, min_samples_leaf=5, random_state=42)
dt.fit(X_tr_dense_ef, y_tr)
acc = accuracy_score(y_v, dt.predict(X_v_dense_ef))
results['Decision Tree'] = acc
print(f"      Val Accuracy: {acc:.4f}")
 
# ── Model 5: Random Forest 
print("\n[5/7] Random Forest ...")
rf = RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_leaf=3,
                             random_state=42, n_jobs=-1)
rf.fit(X_tr_dense_ef, y_tr)
acc = accuracy_score(y_v, rf.predict(X_v_dense_ef))
results['Random Forest'] = acc
print(f"      Val Accuracy: {acc:.4f}")
 
# ── Model 6: K-Nearest Neighbours
print("\n[6/7] K-Nearest Neighbours (k=7) ...")
knn = KNeighborsClassifier(n_neighbors=7, metric='cosine', n_jobs=-1)
knn.fit(X_tr_dense_ef, y_tr)
acc = accuracy_score(y_v, knn.predict(X_v_dense_ef))
results['KNN (k=7)'] = acc
print(f"      Val Accuracy: {acc:.4f}")
 
# ── Model 7: Calibrated LinearSVC 
print("\n[7/7] Calibrated LinearSVC (BEST) ...")
base_svc = LinearSVC(C=0.6, max_iter=3000)
svc_model = CalibratedClassifierCV(base_svc, cv=3, method='sigmoid')
svc_model.fit(X_tr_all, y_tr)
val_preds = svc_model.predict(X_v_all)
acc = accuracy_score(y_v, val_preds)
results['Calibrated LinearSVC ★'] = acc
print(f"Val Accuracy: {acc:.4f}")
 


[1/7] Logistic Regression ...
      Val Accuracy: 0.6338

[2/7] SGD Classifier ...
      Val Accuracy: 0.2295

[3/7] Complement Naive Bayes ...
      Val Accuracy: 0.6572

[4/7] Decision Tree ...
      Val Accuracy: 0.4824

[5/7] Random Forest ...
      Val Accuracy: 0.6104

[6/7] K-Nearest Neighbours (k=7) ...
      Val Accuracy: 0.5781

[7/7] Calibrated LinearSVC (BEST) ...
Val Accuracy: 0.6787


In [9]:

from sklearn.model_selection import GridSearchCV, StratifiedKFold
import time

cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("=" * 60)
print("   HYPERPARAMETER TUNING — TOP 3 MODELS")
print("=" * 60)

hpt_results = {}


print("\n[HPT 1/3] Calibrated LinearSVC ...")
t0 = time.time()

# FIX: sklearn >= 1.2 renamed 'base_estimator' → 'estimator'
# We loop manually over C values to avoid GridSearchCV param naming issues
svc_c_values = [0.1, 0.3, 0.6, 1.0, 2.0, 5.0]
svc_cv_scores = []

for C in svc_c_values:
    fold_accs = []
    for fold_tr, fold_v in cv_inner.split(X_tr_all, y_tr):
        X_f_tr = X_tr_all[fold_tr]
        X_f_v  = X_tr_all[fold_v]
        y_f_tr = y_tr.iloc[fold_tr]
        y_f_v  = y_tr.iloc[fold_v]
        m = CalibratedClassifierCV(LinearSVC(C=C, max_iter=3000), cv=3, method='sigmoid')
        m.fit(X_f_tr, y_f_tr)
        fold_accs.append(accuracy_score(y_f_v, m.predict(X_f_v)))
    mean_acc = np.mean(fold_accs)
    svc_cv_scores.append(mean_acc)
    print(f"    C={C:<5} → CV Accuracy: {mean_acc:.4f}")

best_C_svc = svc_c_values[np.argmax(svc_cv_scores)]
best_svc   = CalibratedClassifierCV(LinearSVC(C=best_C_svc, max_iter=3000), cv=3, method='sigmoid')
best_svc.fit(X_tr_all, y_tr)
svc_preds  = best_svc.predict(X_v_all)
svc_acc    = accuracy_score(y_v, svc_preds)
hpt_results['Calibrated LinearSVC'] = ({'C': best_C_svc}, svc_acc)

print(f"\n  Best C       : {best_C_svc}")
print(f"  Best CV Score: {max(svc_cv_scores):.4f}")
print(f"  Val Accuracy : {svc_acc:.4f}  (baseline was 0.6777)")
print(f"  Time         : {time.time()-t0:.1f}s")


print("\n[HPT 2/3] Logistic Regression ...")
t0 = time.time()

gs_lr = GridSearchCV(
    estimator  = LogisticRegression(solver='saga', max_iter=1000, tol=1e-3),
    param_grid = {'C': [0.01, 0.1, 0.5, 1.0, 3.0, 10.0]},
    cv         = cv_inner,
    scoring    = 'accuracy',
    n_jobs     = -1,
    verbose    = 1
)
gs_lr.fit(X_tr_all, y_tr)

best_lr  = gs_lr.best_estimator_
lr_preds = best_lr.predict(X_v_all)
lr_acc   = accuracy_score(y_v, lr_preds)
hpt_results['Logistic Regression'] = (gs_lr.best_params_, lr_acc)

print(f"  Best Params  : {gs_lr.best_params_}")
print(f"  CV Score     : {gs_lr.best_score_:.4f}")
print(f"  Val Accuracy : {lr_acc:.4f}  (baseline was 0.6338)")
print(f"  Time         : {time.time()-t0:.1f}s")

print("\n[HPT 3/3] Complement Naive Bayes ...")
t0 = time.time()

gs_cnb = GridSearchCV(
    estimator  = ComplementNB(),
    param_grid = {'alpha': [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0]},
    cv         = cv_inner,
    scoring    = 'accuracy',
    n_jobs     = -1,
    verbose    = 1
)
gs_cnb.fit(X_tr_nonneg, y_tr)          # ← non-negative matrix

best_cnb  = gs_cnb.best_estimator_
cnb_preds = best_cnb.predict(X_v_nonneg)
cnb_acc   = accuracy_score(y_v, cnb_preds)
hpt_results['Complement Naive Bayes'] = (gs_cnb.best_params_, cnb_acc)

print(f"  Best Params  : {gs_cnb.best_params_}")
print(f"  CV Score     : {gs_cnb.best_score_:.4f}")
print(f"  Val Accuracy : {cnb_acc:.4f}  (baseline was 0.6572)")
print(f"  Time         : {time.time()-t0:.1f}s")


baseline = {
    'Calibrated LinearSVC'  : 0.6777,
    'Logistic Regression'   : 0.6338,
    'Complement Naive Bayes': 0.6572,
}

print("\n" + "=" * 70)
print("         HPT SUMMARY — BEFORE vs AFTER TUNING")
print("=" * 70)
print(f"  {'Model':<25} {'Before':>8} {'After':>8} {'Delta':>8}  Best Params")
print("-" * 70)
for name, (params, acc) in hpt_results.items():
    base  = baseline[name]
    delta = acc - base
    sign  = "▲" if delta >= 0 else "▼"
    print(f"  {name:<25} {base:>8.4f} {acc:>8.4f} {sign}{abs(delta):>7.4f}  {params}")
print("=" * 70)

best_name = max(hpt_results, key=lambda k: hpt_results[k][1])
best_acc  = hpt_results[best_name][1]
print(f"\n🏆 Best model after HPT : {best_name}  ({best_acc:.4f})")

   HYPERPARAMETER TUNING — TOP 3 MODELS

[HPT 1/3] Calibrated LinearSVC ...
    C=0.1   → CV Accuracy: 0.5977
    C=0.3   → CV Accuracy: 0.6184
    C=0.6   → CV Accuracy: 0.6303
    C=1.0   → CV Accuracy: 0.6329
    C=2.0   → CV Accuracy: 0.6336
    C=5.0   → CV Accuracy: 0.6342

  Best C       : 5.0
  Best CV Score: 0.6342
  Val Accuracy : 0.6709  (baseline was 0.6777)
  Time         : 172.3s

[HPT 2/3] Logistic Regression ...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
  Best Params  : {'C': 10.0}
  CV Score     : 0.6010
  Val Accuracy : 0.6504  (baseline was 0.6338)
  Time         : 19.4s

[HPT 3/3] Complement Naive Bayes ...
Fitting 3 folds for each of 7 candidates, totalling 21 fits
  Best Params  : {'alpha': 1.0}
  CV Score     : 0.6324
  Val Accuracy : 0.6641  (baseline was 0.6572)
  Time         : 0.2s

         HPT SUMMARY — BEFORE vs AFTER TUNING
  Model                       Before    After    Delta  Best Params
-----------------------------------------------

In [10]:
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}") 
# Test shape MUST output (1700, ...)

Train shape: (6823, 7)
Test shape: (1700, 6)


In [11]:
print("=" * 60)
print("   FINAL MODEL SELECTION")
print("=" * 60)

# Scores from Cell 6 (baseline) and Cell 7 (HPT)
baseline_best     = ('Calibrated LinearSVC', 0.6777, {'C': 0.6})
hpt_best_name     = max(hpt_results, key=lambda k: hpt_results[k][1])
hpt_best_params   = hpt_results[hpt_best_name][0]
hpt_best_acc      = hpt_results[hpt_best_name][1]

print(f"\n  Baseline best : Calibrated LinearSVC  →  Val Acc = 0.6777  (C=0.6)")
print(f"  HPT best      : {hpt_best_name:<25} →  Val Acc = {hpt_best_acc:.4f}  {hpt_best_params}")

# --- MODIFIED SECTION: Force baseline to prevent submission score drop ---
print(f"\n  ✅ Forcing Baseline   : Calibrated LinearSVC  (C=0.6)")
use_hpt = False
# -------------------------------------------------------------------------

# ═══════════════════════════════════════════════════════════
#  Rebuild full-data features
# ═══════════════════════════════════════════════════════════
print("\nBuilding full-data features ...")

ef_full = build_extra_features(train['phrase'].values)
ef_test = build_extra_features(test['phrase'].values)

vec_full = TfidfVectorizer(ngram_range=(1,2), max_features=80_000,
                           sublinear_tf=True, min_df=2)
W_full = vec_full.fit_transform(train['phrase'])
W_test = vec_full.transform(test['phrase'])

X_full_all = hstack([W_full, csr_matrix(ef_full)])
X_test_all = hstack([W_test, csr_matrix(ef_test)])

# Non-negative version for CNB (in case CNB won HPT)
mm_full = MinMaxScaler()
ef_full_nonneg = mm_full.fit_transform(ef_full)
ef_test_nonneg = mm_full.transform(ef_test)
X_full_nonneg  = hstack([W_full, csr_matrix(ef_full_nonneg)])
X_test_nonneg  = hstack([W_test, csr_matrix(ef_test_nonneg)])

print(f"  Full train rows : {X_full_all.shape[0]}")
print(f"  Test rows       : {X_test_all.shape[0]}")


# ═══════════════════════════════════════════════════════════
#  Retrain & Predict
# ═══════════════════════════════════════════════════════════
print("\nRetraining on full data ...")

if use_hpt and hpt_best_name == 'Calibrated LinearSVC':
    best_C = hpt_best_params['C']
    final_model = CalibratedClassifierCV(
        LinearSVC(C=best_C, max_iter=3000), cv=3, method='sigmoid'
    )
    final_model.fit(X_full_all, train['sentiment'])
    test_preds = final_model.predict(X_test_all)
    model_desc = f"Calibrated LinearSVC (C={best_C})"

elif use_hpt and hpt_best_name == 'Logistic Regression':
    best_C = hpt_best_params['C']
    final_model = LogisticRegression(C=best_C, solver='saga', max_iter=1000, tol=1e-3)
    final_model.fit(X_full_all, train['sentiment'])
    test_preds = final_model.predict(X_test_all)
    model_desc = f"Logistic Regression (C={best_C})"

elif use_hpt and hpt_best_name == 'Complement Naive Bayes':
    best_alpha = hpt_best_params['alpha']
    final_model = ComplementNB(alpha=best_alpha)
    final_model.fit(X_full_nonneg, train['sentiment'])
    test_preds = final_model.predict(X_test_nonneg)
    model_desc = f"Complement Naive Bayes (alpha={best_alpha})"

else:
    # Fallback: original best baseline
    final_model = CalibratedClassifierCV(
        LinearSVC(C=0.6, max_iter=3000), cv=3, method='sigmoid'
    )
    final_model.fit(X_full_all, train['sentiment'])
    test_preds = final_model.predict(X_test_all)
    model_desc = "Calibrated LinearSVC (C=0.6) [baseline]"

print(f"  Model used : {model_desc}")



labels = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

submission = pd.DataFrame({
    'id'       : test['id'],
    'sentiment': test_preds
})
submission.to_csv('submission.csv', index=False)

print("\n" + "=" * 60)
print("SUBMISSION SAVED → submission.csv")
print("=" * 60)
print(f"  Model        : {model_desc}")
print(f"  Rows         : {len(submission)}  (expected 1700)")
print(f"  ID range     : {submission['id'].min()} → {submission['id'].max()}")
print("\n  Prediction Distribution:")
dist = submission['sentiment'].value_counts().sort_index()
for lbl, count in dist.items():
    pct = count / len(submission) * 100
    bar = '█' * int(pct / 2)
    print(f"    {labels[lbl]:10s} ({lbl}): {count:5d}  ({pct:.1f}%)  {bar}")

print("\n  Sample predictions (first 10):")
print(submission.head(10).to_string(index=False))

print("\nDone! ✓")

   FINAL MODEL SELECTION

  Baseline best : Calibrated LinearSVC  →  Val Acc = 0.6777  (C=0.6)
  HPT best      : Calibrated LinearSVC      →  Val Acc = 0.6709  {'C': 5.0}

  ✅ Forcing Baseline   : Calibrated LinearSVC  (C=0.6)

Building full-data features ...
  Full train rows : 6823
  Test rows       : 1700

Retraining on full data ...
  Model used : Calibrated LinearSVC (C=0.6) [baseline]

SUBMISSION SAVED → submission.csv
  Model        : Calibrated LinearSVC (C=0.6) [baseline]
  Rows         : 1700  (expected 1700)
  ID range     : 0 → 1699

  Prediction Distribution:
    Negative   (0):   809  (47.6%)  ███████████████████████
    Neutral    (1):    37  (2.2%)  █
    Positive   (2):   854  (50.2%)  █████████████████████████

  Sample predictions (first 10):
 id  sentiment
  0          2
  1          2
  2          2
  3          2
  4          0
  5          0
  6          0
  7          2
  8          2
  9          2

Done! ✓
